In [9]:
import joblib
import numpy as np
import pandas as pd
import librosa
import os

In [10]:
model_queen_presence = joblib.load("queen_presence_model.pkl")
model_queen_acceptance = joblib.load("queen_acceptance_model.pkl")
model_anomaly = joblib.load("bee_sound_anomaly_model.pkl")


In [11]:


# Function to extract audio features
def extract_audio_features(file_path, n_mfcc=13):
    y, sr = librosa.load(file_path, sr=22050)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfccs_mean = np.mean(mfccs.T, axis=0)
    return mfccs_mean


audio_path = input("Enter the full path of the audio clip (.wav): ").strip('"')

# You can adjust this section to load inputs from GUI/CSV/file
hive_temp = float(input("Enter hive temperature: "))
hive_humidity = float(input("Enter hive humidity: "))
weather_temp = float(input("Enter weather temperature: "))
weather_humidity = float(input("Enter weather humidity: "))

# Extract MFCC features from audio
mfcc_features = extract_audio_features(audio_path)

# Combine manual input with audio features into a single row for prediction
features = np.concatenate(([hive_temp, hive_humidity, weather_temp, weather_humidity], mfcc_features))
features_df = pd.DataFrame([features], columns=[
    "hive temp", "hive humidity", "weather temp", "weather humidity"
] + [f"mfcc_{i+1}" for i in range(len(mfcc_features))])

# === PREDICTIONS ===
pred_qp = model_queen_presence.predict(features_df)[0]
pred_qa = model_queen_acceptance.predict(features_df)[0]
pred_anomaly = model_anomaly.predict(features_df)[0]  # -1 = anomaly, 1 = normal

# === OUTPUT ===
print("\n🧠 Predictions:")
print(f"🐝 Queen Presence: {'Yes' if pred_qp == 1 else 'No'}")
print(f"👑 Queen Acceptance Level: {pred_qa}")
print(f"🚨 Anomaly Detected in Sound: {'Yes' if pred_anomaly == -1 else 'No'}")


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- hive humidity
- hive temp
- weather humidity
- weather temp


In [12]:
# Function to extract audio features
def extract_audio_features(file_path, n_mfcc=13):
    y, sr = librosa.load(file_path, sr=22050)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfccs_mean = np.mean(mfccs.T, axis=0)
    return mfccs_mean

# === INPUTS ===
audio_path = input("Enter the full path of the audio clip (.wav): ").strip('"')
hive_temp = float(input("Enter hive temperature: "))
hive_humidity = float(input("Enter hive humidity: "))
weather_temp = float(input("Enter weather temperature: "))
weather_humidity = float(input("Enter weather humidity: "))

# === AUDIO FEATURES ===
mfcc_features = extract_audio_features(audio_path)

# === Combined Feature Vector for Model 1 & 2 ===
all_features = np.concatenate(([hive_temp, hive_humidity, weather_temp, weather_humidity], mfcc_features))
all_feature_names = ["hive temp", "hive humidity", "weather temp", "weather humidity"] + [f"mfcc_{i+1}" for i in range(len(mfcc_features))]
features_df_all = pd.DataFrame([all_features], columns=all_feature_names)

# === MFCC-only Feature Vector for Anomaly Model ===
features_df_mfcc_only = pd.DataFrame([mfcc_features], columns=[f"mfcc_{i+1}" for i in range(len(mfcc_features))])

# === PREDICTIONS ===
pred_qp = model_queen_presence.predict(features_df_all)[0]
pred_qa = model_queen_acceptance.predict(features_df_all)[0]
pred_anomaly = model_anomaly.predict(features_df_mfcc_only)[0]  # -1 = anomaly, 1 = normal

# === OUTPUT ===
print("\n🧠 Predictions:")
print(f"🐝 Queen Presence: {'Yes' if pred_qp == 1 else 'No'}")
print(f"👑 Queen Acceptance Level: {pred_qa}")
print(f"🚨 Anomaly Detected in Sound: {'Yes' if pred_anomaly == -1 else 'No'}")



🧠 Predictions:
🐝 Queen Presence: Yes
👑 Queen Acceptance Level: 2
🚨 Anomaly Detected in Sound: No
